# ARC-AGI-3: Duck harness + arc3 tricks (A/B)

Harness: TAAF/Tufa Labs Duck (CC0 bundle); model Qwen3.8-27B-FP8 (Apache 2.0); wrapper from the public notebook foysalemonshanto/lb-9-arc3-duck-v12-with-qwen-3-8-27b.

**Both arms** get the throughput fixes: bounded analysis turns (`MAX_OUTPUT` 3072, `TOOL_STEPS` 12, both shipped as unlimited), a 32768 served context so vLLM holds ~2x the concurrent sequences, and a per-game budget derived from the time actually left in the 9 h kernel with a deadline that also holds in submission mode.

**Arm under test** (odd passes): a stall RESET after 80 actions with no level change, `UNDO`/ACTION7 made reachable, and a no-op guard signature that ignores timer strips.

**Save run:** 6 public games we never taught x 6 passes = 3 per arm. Output: `arc3_ab_report.md` / `.json`. **Competition rerun:** every hidden game plays with the arm on.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
ARC3_SERVING_PATCHES = [
    [
        "VLLM_MAX_MODEL_LEN = 65536",
        "VLLM_MAX_MODEL_LEN = 32768"
    ],
    [
        "ANALYZER_CONTEXT_WINDOW = 32768",
        "ANALYZER_CONTEXT_WINDOW = 24576"
    ],
    [
        "'LOCAL_ANALYZER_MAX_OUTPUT': '0'",
        "'LOCAL_ANALYZER_MAX_OUTPUT': '3072'"
    ],
    [
        "'LOCAL_ANALYZER_TOOL_STEPS': '0'",
        "'LOCAL_ANALYZER_TOOL_STEPS': '12'"
    ]
]

import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        # arc3 item 2: bound the runaway turn and stop paying for a served context
        # window the agent never asks for (both shipped as unlimited / 2x oversized).
        for _old, _new in ARC3_SERVING_PATCHES:
            if _old in command:
                command = command.replace(_old, _new)
                print('arc3 serving patch:', _old, '->', _new, flush=True)
        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# arc3 tricks: write the add-on, choose the arm policy, install before the benchmark is unpickled.
import importlib

ARC3_TRICKS_SOURCE = "\"\"\"arc3_tricks v2 -- three mechanisms grafted onto the Duck harness, A/B-able.\n\nAudited 2026-09-18 (arc3-autoresearch/consult/audit_2026-09-18_*.txt). v1 shipped two\nideas; the audit killed one of them and found a lockout bug in the other, so v2 is:\n\n1. STALL RESET (new, item 3). Duck has no no-progress reset at all -- the only reset\n   trigger in the whole harness is GAME_OVER (solver.py:317-322). Our own lab measured\n   this exact intervention as its single largest hold gain (results.tsv 6caf45c: hold\n   0.1436 -> 0.1541, ft09's first level ever, hold_levels 2 -> 3) and smooth across N.\n   Here: after ARC3_STALL_N actions on one level with no level change, call Duck's own\n   _execute_auto_reset(), clear the agent's stale plan, and tell it what happened.\n\n2. UNDO / ACTION7 (new, item 4). ACTION7 is an undo in the engine and 6 of the 25 public\n   games offer it at level 1, but Duck's action map (action_names.py:7-15) has no entry\n   for it: the harness advertises it in valid_actions and then rejects every attempt.\n   One dict entry each way makes it reachable, and undo is far cheaper than RESET on\n   RHAE because it rewinds instead of replaying.\n\n3. HUD-MASKED GUARD SIGNATURE (v1 trick, repaired). Duck's no-op guard keys on a hash of\n   the entire board (noop_guard.py:16-20), so one ticking counter makes every state look\n   new and the guard can never fire -- measured on the public set, masking the volatile\n   edge lines collapses 89 distinct signatures to 1 on lf52, 120 to 4 on vc33. v2 keeps\n   ONLY the signature half. v1 also demoted \"the board changed\" to False when the diff\n   was HUD-only, which could record a real move as a no-op and then block it forever\n   (a blocked action never executes, so the contradicting evidence never arrives). That\n   half is deleted. The line statistic now needs FRAC=0.8 over MIN_OBS=12 observations\n   and ignores any transition whose diff exceeds SMALL_DIFF of the board, so level\n   changes, resets and animations no longer push edge lines toward the threshold.\n\nv1's nav_moves()/nav_path() sandbox helpers are removed: they misidentified the player on\nbusy boards, rebuilt from the whole history on every call (2.3 s at 400 transitions inside\na 30 s sandbox), and read a raw grid that Duck's own prompt deliberately hides.\n\nArms: ARC3_MODE=ab -> even passes plain, odd passes tricks; on -> always; off -> never.\nPer-run counters land next to the harness artifacts as <run>_arc3.json.\n\ninstall() must run after the TAAF sources are importable and before benchmark_initial.pkl\nis unpickled.\n\"\"\"\nfrom __future__ import annotations\n\nimport inspect\nimport json\nimport os\nimport textwrap\nfrom collections import Counter\nfrom pathlib import Path\n\n# A timer bar moves one cell per tick, so per-cell frequency never identifies it; the\n# statistic has to be per line. Per line it is also what the thing actually is: a strip.\nEDGE = 3          # a volatile row/column must lie within this many lines of the frame edge\nFRAC = 0.8        # ...and change on at least this fraction of small-diff observations\nMIN_OBS = 12      # ...once at least this many such observations exist\nMAX_LINES = 8     # never mask more lines than this\nSMALL_DIFF = 0.05 # ignore transitions whose diff exceeds this fraction of the board\n\nADDENDUM = (\n    \"\\n\\nHarness notes for this run:\\n\"\n    \"- The no-op guard ignores border cells that change on almost every action (timers, step \"\n    \"counters) when it decides whether a board state repeats, so a repeated action that only moved \"\n    \"a timer is blocked before it costs an action.\\n\"\n    \"- If `UNDO` appears in `valid_actions`, it rewinds the last move. Prefer it to replaying a \"\n    \"level: it costs one action and restores the previous state.\\n\"\n    \"- If you spend a long time on one level with no level change, the harness restarts the level \"\n    \"for you and tells you so. A restart costs one action and keeps everything you have learned, so \"\n    \"treat it as a fresh attempt at a level you now understand, not as a loss.\\n\"\n)\n\n_RUN_PATCHES = [\n    # Guard signature: mask volatile border lines so a ticking counter stops hiding repeats.\n    (\"noop_guard_board_sig = board_signature(current_frame.grid) if current_frame is not None else board_signature(())\",\n     \"noop_guard_board_sig = self._arc3_sig(current_frame.grid if current_frame is not None else ())\\n\"\n     \"{I}noop_guard_grid = current_frame.grid if current_frame is not None else ()\", 1),\n    (\"nonlocal terminal_action_result, noop_guard_board_sig, noop_guard_level\",\n     \"nonlocal terminal_action_result, noop_guard_board_sig, noop_guard_level, noop_guard_grid\", 1),\n    # Learn the volatility statistic from every executed action (no demotion of board_changed).\n    ('board_changed=bool(compact_payload.get(\"board_changed\")),',\n     'board_changed=self._arc3_observe(compact_payload, noop_guard_grid, state_path),', 1),\n    ('board_changed=bool(sub_compact.get(\"board_changed\")),',\n     'board_changed=self._arc3_observe(sub_compact, noop_guard_grid, state_path),', 1),\n    (\"noop_guard_board_sig = board_signature(refreshed_frame.grid)\",\n     \"noop_guard_board_sig = self._arc3_sig(refreshed_frame.grid)\\n\"\n     \"{I}noop_guard_grid = refreshed_frame.grid\", 2),\n    (\"blocked_actions.append(action_sig)\",\n     \"blocked_actions.append(action_sig)\\n{I}self._arc3_count('guard_blocks')\", 1),\n    (\"last_result = self._last_action_result or {}\",\n     \"self._arc3_count('guard_blocks')\\n{I}last_result = self._last_action_result or {}\", 1),\n]\n\n_PLAY_PATCHES = [\n    (\"while not self.should_stop():\",\n     \"while not self.should_stop():\\n\"\n     \"{I}    if self._arc3_stalled():\\n\"\n     \"{I}        self._arc3_reset_level()\\n\"\n     \"{I}        continue\", 1),\n]\n\n\ndef arm_for_pass(pass_index: int) -> bool:\n    mode = os.environ.get(\"ARC3_MODE\", \"ab\").strip().lower()\n    if mode == \"on\":\n        return True\n    if mode == \"off\":\n        return False\n    return int(pass_index) % 2 == 1\n\n\ndef _patch(text: str, old: str, new: str, count: int) -> str:\n    \"\"\"Replace `old` exactly `count` times; `{I}` in `new` becomes the matched line's indentation.\"\"\"\n    found = text.count(old)\n    if found != count:\n        raise RuntimeError(f\"arc3_tricks: expected {count} x {old[:60]!r}, found {found}\")\n    out, pos = [], 0\n    while True:\n        at = text.find(old, pos)\n        if at < 0:\n            break\n        line_start = text.rfind(\"\\n\", 0, at) + 1\n        indent = text[line_start:at]\n        indent = indent if not indent.strip() else \"\"\n        out.append(text[pos:at])\n        out.append(new.replace(\"{I}\", indent))\n        pos = at + len(old)\n    out.append(text[pos:])\n    return \"\".join(out)\n\n\ndef _rebind(cls, name, source_owner, patches, extra_globals=None):\n    \"\"\"Re-exec one method's source with text patches applied, and bind it onto cls.\"\"\"\n    src = textwrap.dedent(inspect.getsource(getattr(cls, name)))\n    for old, new, count in patches:\n        src = _patch(src, old, new, count)\n    namespace: dict = {}\n    globals_ = dict(vars(source_owner))\n    globals_.update(extra_globals or {})\n    exec(compile(src, f\"<arc3_{name}>\", \"exec\"), globals_, namespace)\n    setattr(cls, f\"{name}_plain\", getattr(cls, name))\n    setattr(cls, name, namespace[name])\n    return namespace[name]\n\n\ndef install() -> dict:\n    from inference.agent import action_names\n    from inference.agent import tool_agent as ta\n    from inference.framework import solver as solver_mod\n\n    if getattr(ta.ToolAgent, \"_arc3_installed\", False):\n        return {\"installed\": \"already\"}\n\n    # ---- 1. guard signature -------------------------------------------------\n    _rebind(ta.ToolAgent, \"_run_python_tool\", ta, _RUN_PATCHES)\n    patched_run = ta.ToolAgent._run_python_tool\n\n    def _run_python_tool(self, state_path, arguments):\n        self._arc3_count(\"python_calls\")\n        try:\n            return patched_run(self, state_path, arguments)\n        finally:\n            self._arc3_flush()\n\n    def _arc3_count(self, key, amount=1):\n        self.__dict__.setdefault(\"_arc3_stats\", Counter())[key] += amount\n\n    def _arc3_masked_lines(self, shape):\n        \"\"\"Border rows/columns that change on nearly every small-diff action: timers, counters.\n\n        FRAC is deliberately high. A timer ticks on essentially 100% of actions; a player\n        pacing a corridor along an edge row rarely reaches 80% of all actions in a game, and\n        masking its row would let the guard confuse two boards that differ only by where the\n        player stands.\n        \"\"\"\n        rows, cols = shape\n        n = self.__dict__.get(\"_arc3_n\", 0)\n        if n < MIN_OBS or not rows:\n            return frozenset(), frozenset()\n        cached = self.__dict__.get(\"_arc3_mask_cache\")\n        if cached is not None and cached[0] == n:\n            return cached[1]\n        need = FRAC * n\n        row_hits = self.__dict__.get(\"_arc3_rows\", Counter())\n        col_hits = self.__dict__.get(\"_arc3_cols\", Counter())\n        mask_rows = frozenset(r for r, hits in row_hits.items()\n                              if hits >= need and (r < EDGE or r >= rows - EDGE))\n        mask_cols = frozenset(c for c, hits in col_hits.items()\n                              if hits >= need and (c < EDGE or c >= cols - EDGE))\n        if len(mask_rows) + len(mask_cols) > MAX_LINES:\n            mask_rows, mask_cols = frozenset(), frozenset()\n        self.__dict__[\"_arc3_mask_cache\"] = (n, (mask_rows, mask_cols))\n        return mask_rows, mask_cols\n\n    def _arc3_sig(self, grid):\n        if not getattr(self, \"_arc3_on\", False) or not grid:\n            return ta.board_signature(grid)\n        mask_rows, mask_cols = self._arc3_masked_lines((len(grid), len(grid[0])))\n        if not mask_rows and not mask_cols:\n            return ta.board_signature(grid)\n        masked = [\n            [-1 if (r in mask_rows or c in mask_cols) else v for c, v in enumerate(row)]\n            for r, row in enumerate(grid)\n        ]\n        return ta.board_signature(masked)\n\n    def _arc3_observe(self, payload, before, state_path):\n        \"\"\"Learn which cells are volatile. Never changes what the harness thinks happened.\"\"\"\n        env_changed = bool(payload.get(\"board_changed\"))\n        if not getattr(self, \"_arc3_on\", False):\n            return env_changed\n        after_frame, _ = ta.load_runtime_state(state_path)\n        after = after_frame.grid if after_frame is not None else ()\n        if not before or not after or len(before) != len(after):\n            return env_changed\n        cells = len(after) * len(after[0])\n        diff = [\n            (r, c)\n            for r in range(len(after))\n            for c in range(len(after[r]))\n            if r < len(before) and c < len(before[r]) and before[r][c] != after[r][c]\n        ]\n        # A level change, a RESET or a big animation redraws most of the board; counting\n        # those would drag every edge cell toward the threshold for the wrong reason.\n        if len(diff) <= max(1, int(SMALL_DIFF * cells)):\n            self.__dict__[\"_arc3_n\"] = self.__dict__.get(\"_arc3_n\", 0) + 1\n            row_hits = self.__dict__.setdefault(\"_arc3_rows\", Counter())\n            col_hits = self.__dict__.setdefault(\"_arc3_cols\", Counter())\n            for r in {r for r, _ in diff}:\n                row_hits[r] += 1\n            for c in {c for _, c in diff}:\n                col_hits[c] += 1\n            self.__dict__.pop(\"_arc3_mask_cache\", None)\n        return env_changed\n\n    def _arc3_flush(self):\n        path = getattr(self, \"_arc3_stats_path\", None)\n        if path is None:\n            return\n        on = getattr(self, \"_arc3_on\", False)\n        mask_rows, mask_cols = self._arc3_masked_lines((64, 64)) if on else (frozenset(), frozenset())\n        data = {\n            \"arm\": \"tricks\" if on else \"plain\",\n            \"counters\": dict(self.__dict__.get(\"_arc3_stats\", Counter())),\n            \"small_diff_actions\": self.__dict__.get(\"_arc3_n\", 0),\n            \"masked_rows\": sorted(mask_rows),\n            \"masked_cols\": sorted(mask_cols),\n        }\n        try:\n            Path(path).write_text(json.dumps(data, indent=1), encoding=\"utf-8\")\n        except OSError:\n            pass\n\n    for name, fn in [\n        (\"_run_python_tool\", _run_python_tool), (\"_arc3_count\", _arc3_count),\n        (\"_arc3_masked_lines\", _arc3_masked_lines), (\"_arc3_sig\", _arc3_sig),\n        (\"_arc3_observe\", _arc3_observe), (\"_arc3_flush\", _arc3_flush),\n    ]:\n        setattr(ta.ToolAgent, name, fn)\n\n    # ---- 2. UNDO / ACTION7 --------------------------------------------------\n    # Mutate in place: solver.py imported the helper functions, which read these dicts\n    # at call time, so rebinding the names would not reach it.\n    action_names.ENGINE_TO_MODEL_ACTION.setdefault(\"ACTION7\", \"UNDO\")\n    action_names.MODEL_TO_ENGINE_ACTION.setdefault(\"UNDO\", \"ACTION7\")\n\n    # ---- 3. stall reset -----------------------------------------------------\n    session_cls = solver_mod._HarnessGameSession\n\n    def _arc3_stalled(self):\n        limit = int(os.environ.get(\"ARC3_STALL_N\", \"0\") or 0)\n        if not limit or not getattr(self, \"_arc3_on\", False):\n            return False\n        level = int(self.game.current_state.levels_completed)\n        if level != self.__dict__.get(\"_arc3_level\", -1):\n            self.__dict__[\"_arc3_level\"] = level\n            self.__dict__[\"_arc3_mark\"] = self.action_count\n            return False\n        return self.action_count - self.__dict__.get(\"_arc3_mark\", 0) >= limit\n\n    def _arc3_reset_level(self):\n        spent = self.action_count - self.__dict__.get(\"_arc3_mark\", 0)\n        self.__dict__[\"_arc3_mark\"] = self.action_count\n        self._execute_auto_reset()\n        agent = self.analyzer\n        agent._arc3_count(\"stall_resets\")\n        # The plan that wedged us is stale; the evidence behind it is not, so history stays.\n        knowledge = getattr(agent, \"_summarized_knowledge\", None)\n        if isinstance(knowledge, dict):\n            knowledge[\"current_plan\"] = \"\"\n            note = (\n                f\"Level restarted by the harness after {spent} actions with no level change. \"\n                \"The board is back at the start of this level; everything learned still applies.\"\n            )\n            previous = str(knowledge.get(\"recent_findings\") or \"\")\n            knowledge[\"recent_findings\"] = (note if not previous else f\"{previous} | {note}\")[:2000]\n\n    session_cls._arc3_stalled = _arc3_stalled\n    session_cls._arc3_reset_level = _arc3_reset_level\n    _rebind(session_cls, \"play\", solver_mod, _PLAY_PATCHES)\n    plain_play = session_cls.play\n\n    def play(self, *args, **kwargs):\n        on = arm_for_pass(self.pass_index)\n        self._arc3_on = on\n        agent = self.analyzer\n        agent._arc3_on = on\n        agent._arc3_stats_path = Path(str(self.state_path).replace(ta.RUNTIME_STATE_FILENAME, \"arc3.json\"))\n        if on and ADDENDUM not in agent._system_prompt:\n            agent._system_prompt = agent._system_prompt + ADDENDUM\n        agent._arc3_flush()\n        return plain_play(self, *args, **kwargs)\n\n    session_cls.play = play\n    ta.ToolAgent._arc3_installed = True\n    return {\n        \"installed\": True,\n        \"mode\": os.environ.get(\"ARC3_MODE\", \"ab\"),\n        \"stall_n\": os.environ.get(\"ARC3_STALL_N\", \"0\"),\n        \"undo\": action_names.ENGINE_TO_MODEL_ACTION.get(\"ACTION7\"),\n    }\n"
Path('/kaggle/working/arc3_tricks.py').write_text(ARC3_TRICKS_SOURCE, encoding='utf-8')
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
# Save run: even passes plain Duck, odd passes Duck + tricks. Competition rerun: tricks on.
os.environ['ARC3_MODE'] = 'on' if _env_bool('KAGGLE_IS_COMPETITION_RERUN', False) else 'ab'
os.environ['ARC3_STALL_N'] = '80'
import arc3_tricks
arc3_tricks = importlib.reload(arc3_tricks)
print('arc3_tricks:', arc3_tricks.install())


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    # arc3: upstream returns None in submission mode, so nothing ever cancels the
    # benchmark and a hard kernel kill would skip teardown entirely.
    if max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Inline customization hook — Q38 P1-style public evaluation.
#
# Q38 P1 runs the full 25 public ARC-AGI-3 games once each (25 games × 1 pass).
# This override applies only to the public/offline notebook run. Competition reruns
# still replace bm.games from Kaggle's live gateway in the final run cell.

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))
print("Qwen3.8 model path:", os.environ.get("TAAF_QWEN_MODEL_PATH"))

Q38_P1_PUBLIC_GAME_IDS = [
    "ft09-0d8bbf25",
    "sb26-7fbdac44",
    "s5i5-18d95033",
    "re86-8af5384d",
    "m0r0-492f87ba",
    "cn04-2fe56bfb",
]

if not true_submission:
    if len(Q38_P1_PUBLIC_GAME_IDS) < 1 or len(set(Q38_P1_PUBLIC_GAME_IDS)) != len(Q38_P1_PUBLIC_GAME_IDS):
        raise RuntimeError("Public smoke game list must contain unique games.")
    if not bm.games:
        raise RuntimeError("benchmark_initial.pkl contains no template public game.")

    import taaf.game_api

    template_game = bm.games[0]
    arcade_spec = getattr(template_game, "arcade_spec", None)
    if arcade_spec is None:
        arcade_spec = getattr(template_game, "_arcade_spec", None)
    if arcade_spec is None:
        raise RuntimeError(
            "Could not recover the public ArcadeSpec from benchmark_initial.pkl; "
            "cannot construct the 25-game Q38 P1 evaluation set."
        )

    bm.games = [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=arcade_spec)
        for game_id in Q38_P1_PUBLIC_GAME_IDS
    ]
    bm.n_passes = 6
    bm.game_weights = None

    # These already match Q38 P1 in the source notebook; set them explicitly so the
    # intended evaluation configuration is visible and stable.
    if hasattr(bm.solver, "concurrency"):
        bm.solver.concurrency = 18
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        bm.solver.max_runtime_s_per_game = 1800.0

    bm.label = f"{bm.label}-arc3-ab-6g"
    print(f"Public evaluation override: {len(bm.games)} games × {bm.n_passes} pass = {len(bm.games) * bm.n_passes} runs")
    print("Public evaluation concurrency:", getattr(bm.solver, "concurrency", None))
    print("Public per-game runtime cap (s):", getattr(bm.solver, "max_runtime_s_per_game", None))


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    # arc3 item 2c: derive the per-game cap from the time actually left in the kernel, and
    # keep a soft deadline in submission mode so Benchmark.run's teardown always executes.
    import math

    KERNEL_LIMIT_S = float(getattr(target, 'max_runtime_s', 0.0) or 32400.0)
    TEARDOWN_RESERVE_S = 900.0
    _elapsed = time.time() - NOTEBOOK_START_EPOCH
    _remaining = max(600.0, KERNEL_LIMIT_S - _elapsed - TEARDOWN_RESERVE_S)
    _waves = max(1, math.ceil(len(bm.games) * max(1, bm.n_passes) / max(1, int(bm.solver.concurrency))))
    _derived_cap = _remaining / _waves
    if true_submission:
        bm.solver.max_runtime_s_per_game = _derived_cap
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
            seconds=KERNEL_LIMIT_S - TEARDOWN_RESERVE_S)
    else:
        bm.solver.max_runtime_s_per_game = min(bm.solver.max_runtime_s_per_game, _derived_cap)
    print(f'arc3 budget: kernel={KERNEL_LIMIT_S:.0f}s elapsed={_elapsed:.0f}s remaining={_remaining:.0f}s '
          f'games={len(bm.games)} passes={bm.n_passes} waves={_waves} '
          f'per_game={bm.solver.max_runtime_s_per_game:.0f}s soft_end={soft_end}', flush=True)

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# A/B report: plain Duck (even passes) vs Duck + arc3 tricks (odd passes)
if not true_submission:
    import collections

    bench = json.loads((WORKING_DIR / 'benchmark.json').read_text(encoding='utf-8'))
    seen = collections.Counter()
    rows = []
    for run in bench.get('game_runs', []):
        gid = run.get('game_id', '')
        p = seen[gid]
        seen[gid] += 1
        stats_files = sorted((WORKING_DIR / 'artifacts').glob(f'{gid}*_p{p}_arc3.json'))
        stats = json.loads(stats_files[0].read_text()) if stats_files else {}
        tokens = re.search(r'tokens=(\d+)', str(run.get('solver_note', '')))
        rows.append({
            'game': gid.split('-')[0],
            'pass': p,
            'arm': stats.get('arm', 'tricks' if p % 2 else 'plain'),
            'levels': run.get('levels_completed', 0) or 0,
            'n_levels': run.get('number_of_levels'),
            'actions': len(run.get('history') or []),
            'actions_per_level': run.get('actions_per_level'),
            'score': round(float(run.get('final_score') or 0), 3),
            'state': run.get('state'),
            'seconds': round(float(run.get('final_wallclock_seconds') or 0)),
            'tokens': int(tokens.group(1)) if tokens else None,
            'counters': stats.get('counters', {}),
            'masked_rows': stats.get('masked_rows'),
            'masked_cols': stats.get('masked_cols'),
        })

    def summarize(arm):
        rs = [r for r in rows if r['arm'] == arm]
        actions = sum(r['actions'] for r in rs)
        tokens = sum(r['tokens'] or 0 for r in rs)
        return {
            'runs': len(rs),
            'mean_score': round(sum(r['score'] for r in rs) / max(1, len(rs)), 3),
            'levels': sum(r['levels'] for r in rs),
            'runs_with_a_level': sum(1 for r in rs if r['levels']),
            'actions': actions,
            'tokens': tokens,
            'tokens_per_action': round(tokens / max(1, actions)),
            'seconds': sum(r['seconds'] for r in rs),
            'stall_resets': sum(r['counters'].get('stall_resets', 0) for r in rs),
            'guard_blocks': sum(r['counters'].get('guard_blocks', 0) for r in rs),
        }

    summary = {arm: summarize(arm) for arm in ('plain', 'tricks')}
    per_game = {}
    for game in sorted({r['game'] for r in rows}):
        for arm in ('plain', 'tricks'):
            rs = [r for r in rows if r['game'] == game and r['arm'] == arm]
            per_game.setdefault(game, {})[arm] = {
                'levels': [r['levels'] for r in rs],
                'actions': [r['actions'] for r in rs],
                'mean_score': round(sum(r['score'] for r in rs) / max(1, len(rs)), 3),
            }
    wins = sum(1 for g, d in per_game.items()
               if sum(d['tricks']['levels']) > sum(d['plain']['levels']))
    losses = sum(1 for g, d in per_game.items()
                 if sum(d['tricks']['levels']) < sum(d['plain']['levels']))
    verdict = ('tricks ahead on levels' if wins > losses else
               'plain ahead on levels' if losses > wins else 'tie on levels')
    report = {'label': bench.get('label'), 'summary': summary, 'per_game': per_game,
              'games_won_by_tricks': wins, 'games_lost_by_tricks': losses, 'verdict': verdict,
              'runs': rows,
              'note': 'Decide on levels cleared across 3 passes per arm, not on one game.'}
    (WORKING_DIR / 'arc3_ab_report.json').write_text(json.dumps(report, indent=1), encoding='utf-8')

    lines = ['# arc3 A/B report', '', f'verdict: {verdict} ({wins} games better, {losses} worse)', '',
             '| arm | runs | mean score | levels | runs with a level | actions | tokens/action | stall resets | guard blocks |',
             '|---|---|---|---|---|---|---|---|---|']
    for arm, s in summary.items():
        lines.append(f"| {arm} | {s['runs']} | {s['mean_score']} | {s['levels']} | {s['runs_with_a_level']} "
                     f"| {s['actions']} | {s['tokens_per_action']} | {s['stall_resets']} | {s['guard_blocks']} |")
    lines += ['', '| game | plain levels | tricks levels | plain actions | tricks actions | plain score | tricks score |',
              '|---|---|---|---|---|---|---|']
    for game, d in per_game.items():
        lines.append(f"| {game} | {d['plain']['levels']} | {d['tricks']['levels']} | {d['plain']['actions']} "
                     f"| {d['tricks']['actions']} | {d['plain']['mean_score']} | {d['tricks']['mean_score']} |")
    lines += ['', '| game | pass | arm | levels | actions | score | state | s | tokens | counters | masked rows/cols |',
              '|---|---|---|---|---|---|---|---|---|---|---|']
    for r in sorted(rows, key=lambda r: (r['game'], r['pass'])):
        lines.append(f"| {r['game']} | {r['pass']} | {r['arm']} | {r['levels']}/{r['n_levels']} | {r['actions']} "
                     f"| {r['score']} | {r['state']} | {r['seconds']} | {r['tokens']} | {r['counters']} "
                     f"| {r['masked_rows']}/{r['masked_cols']} |")
    (WORKING_DIR / 'arc3_ab_report.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print('\n'.join(lines))


In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")